# Clasificación de imágenes con una CNN usando CIFAR-10

## Objetivo
Construir y evaluar una red neuronal convolucional para clasificar imágenes en 10 categorías usando CIFAR-10.

## Dataset
CIFAR-10 contiene 60.000 imágenes a color de 32 × 32 píxeles: 50.000 para entrenamiento original y 10.000 para prueba. Se separará una parte del conjunto original de entrenamiento para validación, sin usar el conjunto de prueba durante el entrenamiento.

## Clases
Avión, automóvil, pájaro, gato, ciervo, perro, rana, caballo, barco y camión.

## 1. Importar librerías

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 2. Cargar y explorar el dataset

In [ ]:
class_names = [
    "avion", "automovil", "pajaro", "gato", "ciervo",
    "perro", "rana", "caballo", "barco", "camion"
]

(x_train_original, y_train_original), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_original = y_train_original.flatten()
y_test = y_test.flatten()

print("Imágenes originales de entrenamiento:", x_train_original.shape)
print("Etiquetas de entrenamiento:", y_train_original.shape)
print("Imágenes de prueba:", x_test.shape)
print("Etiquetas de prueba:", y_test.shape)
print("Cantidad de clases:", len(class_names))

In [ ]:
plt.figure(figsize=(12, 7))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_train_original[i])
    plt.title(class_names[y_train_original[i]])
    plt.axis("off")
plt.suptitle("Ejemplos del dataset CIFAR-10")
plt.tight_layout()
plt.show()

In [ ]:
counts = np.bincount(y_train_original, minlength=len(class_names))
plt.figure(figsize=(10, 4))
plt.bar(class_names, counts)
plt.title("Distribución de clases en el conjunto original de entrenamiento")
plt.ylabel("Cantidad de imágenes")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 3. División y preprocesamiento
Se reserva el 10 % del conjunto original de entrenamiento para validación. El conjunto de prueba oficial se mantiene separado. Los píxeles se normalizan entre 0 y 1.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_original,
    y_train_original,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_original
)

x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Entrenamiento:", x_train.shape, y_train.shape)
print("Validación:", x_val.shape, y_val.shape)
print("Prueba:", x_test.shape, y_test.shape)
print("Rango de píxeles normalizados:", x_train.min(), "a", x_train.max())

## 4. Data Augmentation y construcción de la CNN

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10)
], name="aumento_datos")

model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.20),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Entrenamiento

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "cnn_cifar10.keras", monitor="val_loss", save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
    )
]

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

## 6. Gráficas del entrenamiento

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Entrenamiento")
plt.plot(history.history["val_accuracy"], label="Validación")
plt.title("Precisión por época")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Entrenamiento")
plt.plot(history.history["val_loss"], label="Validación")
plt.title("Pérdida por época")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 7. Evaluación final en el conjunto de prueba

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Loss de prueba: {test_loss:.4f}")
print(f"Accuracy de prueba: {test_accuracy:.4f}")

y_prob = model.predict(x_test, batch_size=128, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("\nReporte de clasificación:")
print(classification_report(
    y_test, y_pred, target_names=class_names, digits=4, zero_division=0
))

## 8. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(11, 11))
display.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Matriz de confusión - CIFAR-10")
plt.tight_layout()
plt.show()

## 9. Al menos 10 predicciones con imágenes

In [ ]:
plt.figure(figsize=(15, 8))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow((x_test[i] * 255).astype("uint8"))
    real = class_names[y_test[i]]
    pred = class_names[y_pred[i]]
    confianza = float(np.max(y_prob[i])) * 100
    plt.title(f"Real: {real}\nPred: {pred} ({confianza:.1f}%)")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 10. Guardar el modelo y las clases

In [ ]:
os.makedirs("modelo", exist_ok=True)
model.save("modelo/cnn_cifar10.keras")

with open("modelo/class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("Modelo guardado en modelo/cnn_cifar10.keras")
print("Clases guardadas en modelo/class_names.json")

## 11. Descargar los archivos del modelo


In [ ]:
import shutil
shutil.make_archive("modelo_cnn_cifar10", "zip", "modelo")
from google.colab import files
files.download("modelo_cnn_cifar10.zip")